# Evaluation

All retrieval evaluation for the thesis. It runs entirely from the two `experiment.jsonl` files produced by the pipeline, with no GPU. The lexical baselines (BoW and TF-IDF) are computed here rather than in the pipeline, so they are fit on the matched set — the exact summaries that are scored.

**Setup:**
- 1. Run the full pipeline.ipynb twice, once for the non-pseudonymized arm and once for the pseudonymized arm. 
- 2. Set the two folder paths in the setup cell below to your preferred pseudonymized and non-pseudonymzed experiment names.
- 3. Select the pipeline `venv` as this notebook's kernel.
- 4. Run this notebook top to bottom, only one time. We calculate all evaluation metrics for both arms. 


**What is evaluated.** Each summary is both a query and a candidate. A retrieval is correct when it returns another summary of the same work (same `wikidata_id`), and a summary never retrieves itself. The representations scored are:
- **Lexical baselines:** BoW and TF-IDF, over the full text and over the event triggers.
- **Dense encoders:** E5-Mistral and Qwen3-0.6B embed all six conditions (`raw_text`, `events_only`, `temporal`, `causal`, `temporal_causal_independent`, `temporal_causal_joint`); StoryEmbed embeds `raw_text` only.

Note that all dense and sparse vectors are L2-normalized, so cosine similarity is the dot product everywhere.

**Sections:**
1. **Matched intersection:** restrict both arms to the same summaries (I').
2. **Baselines:** compute BoW and TF-IDF (full text and event triggers) on the matched set.
3. **Cosine similarity:** pairwise similarity matrices, per arm.
4. **Overall Retrieval Performance:** overall retrieval, non-pseudonymized vs pseudonymized.
5. **Incremental contribution:** effect of adding temporal and causal structure.
6. **Tversky diagnostic:** retrieval from event-set overlap alone.

In [1]:
# Imports
import os, sys, re, json, collections
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display, Markdown
import random

# This notebook lives in notebooks/, so the project root is the parent of the working directory.
ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_EXPERIMENTS = ROOT / "data" / "experiments"

# The two pipeline runs to evaluate. BOTH are required.
NONANON_DIR = DATA_EXPERIMENTS / "experiment_test_9385_20260609_1013"        # non-pseudonymized arm
ANON_DIR    = DATA_EXPERIMENTS / "anon_experiment_test_9385_20260609_1013"   # pseudonymized arm

# Stop early unless both arms are set and each has an experiment.jsonl.
arms = {"non-pseudonymized (NONANON_DIR)": NONANON_DIR, "pseudonymized (ANON_DIR)": ANON_DIR}
missing_arms = [label for label, directory in arms.items() if directory is None]
if missing_arms:
    raise ValueError("Evaluation requires BOTH dataset paths. Missing: " + ", ".join(missing_arms))
for label, directory in arms.items():
    if not (Path(directory) / "experiment.jsonl").exists():
        raise FileNotFoundError(f"{label}: {Path(directory) / 'experiment.jsonl'} not found. "
                                f"Run the full pipeline for this arm before evaluating.")

# Print out the confirmation
print("Evaluation setup OK")
print(f"  non-pseudonymized : {NONANON_DIR}")
print(f"  pseudonymized     : {ANON_DIR}")

Evaluation setup OK
  non-pseudonymized : /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/experiments/experiment_test_9385_20260609_1013
  pseudonymized     : /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/experiments/anon_experiment_test_9385_20260609_1013


### 1. Matched intersection of the two arms (non-pseudonymized and pseudonymized)

**Goal:** Restrict both arms to the same set of summaries, so every later table compares the non-pseudonymized and pseudonymized runs on an identical population.

**How:** Read the `(wikidata_id, summary_id)` keys from both `experiment.jsonl` files, keep only the keys present in both arms, then re-enforce the floor of at least 2 summaries per work. Both `experiment.jsonl` files are then rewritten in place to this matched set (called I'). 

**Important:** This overwrites both `experiment.jsonl` files because this intersection is the single and absolute source of truth of the thesis, so keep backups before running this notebook if needed. 

In [2]:
# NONANON_DIR and ANON_DIR are defined once in the setup cell at the top of this notebook.
experiment_files = {"non-anon": NONANON_DIR / "experiment.jsonl", "anon": ANON_DIR / "experiment.jsonl"}
id_pattern = re.compile(r'"wikidata_id":\s*"([^"]*)",\s*"summary_id":\s*"([^"]*)"')

# Read the (wikidata_id, summary_id) of every row; both ids sit deterministically within the first 160 characters.
def read_summary_keys(path):
    keys = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            match = id_pattern.search(line[:160])
            if match: keys.add((match.group(1), match.group(2)))
    return keys

# Build I': keys present in BOTH arms, then keep only works that still have >=2 summaries, so a relevance cluster has at least 2 summaries to be eligible for retrieval.
nonanon_keys, anon_keys = read_summary_keys(experiment_files["non-anon"]), read_summary_keys(experiment_files["anon"])
shared_keys = nonanon_keys & anon_keys
summaries_per_work = collections.Counter(wid for wid, _ in shared_keys)
Iprime = {(wid, sid) for (wid, sid) in shared_keys if summaries_per_work[wid] >= 2}
print(f"non-anon              : {len(nonanon_keys):>5} summaries / {len({wid for wid, _ in nonanon_keys})} stories")
print(f"anon                  : {len(anon_keys):>5} summaries / {len({wid for wid, _ in anon_keys})} stories")
print(f"intersection          : {len(shared_keys):>5} summaries   (non-anon -{len(nonanon_keys)-len(shared_keys)}, anon -{len(anon_keys)-len(shared_keys)})")
print(f"matched set I' (>=2)   : {len(Iprime):>5} summaries / {len({wid for wid, _ in Iprime})} stories   (cluster removal -{len(shared_keys)-len(Iprime)})\n")

# Rewrite each file in place, keeping only the I' rows: stream to a .tmp file, then atomic os.replace so a mid-run failure can never leave a half-written experiment.jsonl.
for arm_label, experiment_path in experiment_files.items():
    total_rows = kept_rows = 0
    tmp_path = experiment_path.with_suffix(experiment_path.suffix + ".tmp")
    with open(experiment_path, encoding="utf-8") as infile, open(tmp_path, "w", encoding="utf-8") as outfile:
        for line in infile:
            match = id_pattern.search(line[:160])
            if not match: continue
            total_rows += 1
            if (match.group(1), match.group(2)) in Iprime:
                outfile.write(line if line.endswith("\n") else line + "\n"); kept_rows += 1
    os.replace(tmp_path, experiment_path)
    print(f"{arm_label:9s}: {experiment_path.name}  {total_rows} -> {kept_rows} rows   (dropped {total_rows-kept_rows})")
print("\nBoth files now contain ONLY the matched intersection I'.")

non-anon              :  4315 summaries / 1658 stories
anon                  :  4315 summaries / 1658 stories
intersection          :  4315 summaries   (non-anon -0, anon -0)
matched set I' (>=2)   :  4315 summaries / 1658 stories   (cluster removal -0)

non-anon : experiment.jsonl  4315 -> 4315 rows   (dropped 0)
anon     : experiment.jsonl  4315 -> 4315 rows   (dropped 0)

Both files now contain ONLY the matched intersection I'.


### 2. Baselines (BoW + TF-IDF)

**Goal:** Compute the lexical retrieval baselines (Bag-of-Words and TF-IDF) for each summary from `row["text"]` only. Both are stored as sparse, L2-normalized vectors, so retrieval uses the same `a · b` (dot-product) code path as the dense embeddings.

We run this locally in the notebook, as it is a cheap CPU operation, so it doesn't need GPU nor cluster.

**Preprocessing:** Each summary's text is lowercased, tokenized with scikit-learn's default word pattern (`\b\w\w+\b`), filtered through NLTK English stopwords, and WordNet-lemmatized, then vectorized with `CountVectorizer` (BoW) and `TfidfVectorizer` (TF-IDF). This matches the thesis (Section 3.3.3, following Chaturvedi): BoW and TF-IDF are surface-overlap baselines over the original, non-linearized text. TF-IDF keeps its scikit-learn defaults otherwise (`smooth_idf=True`, `sublinear_tf=False`, `norm='l2'`); the BoW vectors are L2-normalized explicitly.

**Schema** added to each row of `experiment.jsonl`:

```jsonc
"baselines": {
  "bow":   {"model_id": "sklearn.CountVectorizer", "task": "raw_text", "dim": V, "vector": {"indices": [...], "values": [...]}},
  "tfidf": {"model_id": "sklearn.TfidfVectorizer", "task": "raw_text", "dim": V, "vector": {"indices": [...], "values": [...]}}
}
```

**Note:** The learned vocabularies (constant across rows) are saved once to `$EXPERIMENT_DIR/baselines/vocab.json` as `{"bow": [...], "tfidf": [...]}`, where each list position is the feature index and the value is the word, for error-analysis interpretability. This step overwrites `experiment.jsonl` in place.

In [3]:
# Raw-text lexical baselines (BoW + TF-IDF), recomputed AFTER the matched intersection so both arms are
# fit on the same I'=4315 texts (fixes the TF-IDF drift from fitting on the larger pre-intersection corpus).
# Streams experiment.jsonl twice (fit, then write) so the multi-GB inline embeddings never all sit in memory.
import nltk
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download("wordnet", quiet=True)
nltk.download("stopwords", quiet=True)
lemmatizer    = WordNetLemmatizer()
stop_words    = set(stopwords.words("english"))
baseTokenizer = CountVectorizer().build_tokenizer()

# Kept at module scope on purpose: the event-trigger baseline cell (7.1) asserts lemma_tokenizer exists.
def lemma_tokenizer(doc):
    return [lemmatizer.lemmatize(t) for t in baseTokenizer(doc.lower()) if t not in stop_words]

def _sparse_row(matrix, i):
    """Return {indices, values} for row i of a csr_matrix."""
    row = matrix.getrow(i)
    return {"indices": row.indices.tolist(), "values": row.data.tolist()}

def compute_text_baselines(experiment_dir):
    """Fit BoW + TF-IDF on this arm's raw summary text (experiment.jsonl, already restricted to I'),
    write baselines.bow / baselines.tfidf into each row, and save vocab.json. The event-trigger
    baselines are added additively by the next cell."""
    exp        = experiment_dir / "experiment.jsonl"
    vocab_path = experiment_dir / "baselines" / "vocab.json"
    vocab_path.parent.mkdir(parents=True, exist_ok=True)
    assert exp.exists(), f"{exp} not found."

    # Pass 1 - collect raw texts in file order (embeddings ignored).
    texts = []
    with open(exp, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                texts.append(json.loads(line)["text"])

    # Fit BoW (L2-normalized) and TF-IDF (L2 built-in) on the matched set.
    bow_vec   = CountVectorizer(tokenizer=lemma_tokenizer, token_pattern=None)
    tfidf_vec = TfidfVectorizer(tokenizer=lemma_tokenizer, token_pattern=None)
    X_bow   = normalize(bow_vec.fit_transform(texts).astype(np.float64), norm="l2", axis=1, copy=False)
    X_tfidf = tfidf_vec.fit_transform(texts)

    vocab_path.write_text(json.dumps({
        "bow":   bow_vec.get_feature_names_out().tolist(),
        "tfidf": tfidf_vec.get_feature_names_out().tolist(),
    }, ensure_ascii=False), encoding="utf-8")

    dim_bow, dim_tfidf = X_bow.shape[1], X_tfidf.shape[1]

    # Pass 2 - stream again, set raw-text baselines per row, atomic rewrite.
    tmp = exp.with_suffix(exp.suffix + ".tmp"); i = 0
    with open(exp, encoding="utf-8") as fin, open(tmp, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            r = json.loads(line)
            r["baselines"] = {
                "bow":   {"model_id": "sklearn.CountVectorizer", "task": "raw_text", "dim": dim_bow,   "vector": _sparse_row(X_bow, i)},
                "tfidf": {"model_id": "sklearn.TfidfVectorizer", "task": "raw_text", "dim": dim_tfidf, "vector": _sparse_row(X_tfidf, i)},
            }
            fout.write(json.dumps(r, ensure_ascii=False) + "\n"); i += 1
    os.replace(tmp, exp)

    nnz_bow, nnz_tfidf = np.diff(X_bow.indptr), np.diff(X_tfidf.indptr)
    print(f"[{experiment_dir.name}] raw-text baselines on I'={len(texts)} rows")
    print(f"  BoW   vocab={dim_bow:>7,}  nnz mean={nnz_bow.mean():.1f}  p95={int(np.quantile(nnz_bow, 0.95))}  max={nnz_bow.max()}")
    print(f"  TFIDF vocab={dim_tfidf:>7,}  nnz mean={nnz_tfidf.mean():.1f}  p95={int(np.quantile(nnz_tfidf, 0.95))}  max={nnz_tfidf.max()}")
    print(f"  vocab -> {vocab_path}")

for arm_dir in (NONANON_DIR, ANON_DIR):
    compute_text_baselines(arm_dir)


[experiment_test_9385_20260609_1013] raw-text baselines on I'=4315 rows
  BoW   vocab= 35,407  nnz mean=126.4  p95=258  max=399
  TFIDF vocab= 35,407  nnz mean=126.4  p95=258  max=399
  vocab -> /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/experiments/experiment_test_9385_20260609_1013/baselines/vocab.json
[anon_experiment_test_9385_20260609_1013] raw-text baselines on I'=4315 rows
  BoW   vocab= 25,621  nnz mean=111.4  p95=232  max=386
  TFIDF vocab= 25,621  nnz mean=111.4  p95=232  max=386
  vocab -> /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/experiments/anon_experiment_test_9385_20260609_1013/baselines/vocab.json


**Goal:** Randomly check some of the baseline results to make sure the code worked as intended

In [5]:
# Choose which arm's examples to preview
ARM = NONANON_DIR        # NONANON_DIR or ANON_DIR

# Define constants for sampling and previewing the BoW and TF-IDF vectors for a few random summaries
SAMPLE_SEED = 12         # change to see different samples
N_SAMPLES   = 3         # number of summaries to sample for inspection of the BoW and TF-IDF vectors
TOP_K       = 5         # number of top non-zero entries to show for each vector, for better readability
PREVIEW     = 500       # number of characters of the raw text to show in the preview, for better readability

# Load the experiment data again 
# (stream from the selected arm and drop embeddings so the multi-GB file does not blow up memory)
rows  = [{k: v for k, v in json.loads(l).items() if k != "embeddings"}
         for l in open(ARM / "experiment.jsonl", encoding="utf-8") if l.strip()]
vocab = json.loads((ARM / "baselines" / "vocab.json").read_text(encoding="utf-8"))

# Take a random sample of rows for inspection
rng    = random.Random(SAMPLE_SEED)
sample = rng.sample(rows, k=min(N_SAMPLES, len(rows)))

# For each sampled row, create a Markdown string with the necessary data for better readbility
for r in sample:
    md = [f"### {r['wikidata_id']} / {r['summary_id']} ({r['lang']})"]
    md.append(f"**text** (first {PREVIEW} chars):")
    md.append("```\n" + r["text"][:PREVIEW] + ("..." if len(r["text"]) > PREVIEW else "") + "\n```")
    for method in ("bow", "tfidf"):
        b = r["baselines"][method]
        pairs = sorted(zip(b["vector"]["indices"], b["vector"]["values"]),
                       key=lambda iv: -iv[1])[:TOP_K]
        rendered = ", ".join(f"`{vocab[method][i]}`={v:.4f}" for i, v in pairs)
        md.append(f"**top-{TOP_K} {method}**: {rendered}")
    display(Markdown("\n\n".join(md)))


### 798650 / it (it)

**text** (first 500 chars):

```
Michael O'Brien is a 16-year-old Irish thug from Chicago. Although most of his crimes are simple acts of vandalism, the young man decides to steal a briefcase containing drugs from his rival, Paco Moreno. Everything goes wrong: Michael's best friend, Carl Brenner, is killed, and Michael, while trying to escape the police, accidentally runs over Paco's brother, an 8-year-old boy who was on his way home. Michael is sent to a juvenile detention center, where he meets a young boy, Barry Horowitz, wh...
```

**top-5 bow**: `michael`=0.5208, `paco`=0.4006, `reformatory`=0.2404, `horowitz`=0.2003, `escape`=0.1202

**top-5 tfidf**: `paco`=0.5282, `michael`=0.4465, `horowitz`=0.3133, `reformatory`=0.3130, `lofgren`=0.1315

### 26921375 / fr (fr)

**text** (first 500 chars):

```
In 2007, a jewelry store robbery in Madrid went badly wrong. The three criminals in the store ran away. The driver who was waiting for them, Curro, has been arrested. He doesn't denounce his accomplices. An employee died of her injuries. The jeweler remains in a permanent vegetative state.
For eight years, Curro served his sentence. In prison, he is sometimes allowed to have sex with his friend, Ana. That's how they have a son. Ana works without enthusiasm in the modest bar that her brother, Jua...
```

**top-5 bow**: `curro`=0.4663, `josé`=0.4041, `ana`=0.3419, `juanjo`=0.2176, `bar`=0.1554

**top-5 tfidf**: `curro`=0.5917, `josé`=0.4402, `ana`=0.3623, `juanjo`=0.2943, `triana`=0.1764

### 43303311 / de (de)

**text** (first 500 chars):

```
In the first clip, you can see the reenactment of the events of the first known murder of the so-called Zodiac killer on December 20, 1968. It is portrayed quite factually, so that, for example, the victim Betty Lou Jensen is shot while fleeing a few meters from the car.
After that, the strip jumps to the present day and focuses on the young couple Zoe & Mick Branson. They are poor, live in a trailer, and earn money by cutting hair and mowing lawns. Mick reports that he and his buddy Harvey auct...
```

**top-5 bow**: `zoe`=0.3214, `harvey`=0.2571, `mick`=0.2571, `zodiac`=0.2571, `find`=0.1928

**top-5 tfidf**: `zoe`=0.4056, `zodiac`=0.3511, `harvey`=0.3045, `mick`=0.3014, `rented`=0.1442

### 2.a Event-trigger lexical baselines (BoW + TF-IDF over trigger words)

**Goal:** Compute two event-only lexical baselines — BoW and TF-IDF over each summary's event-trigger words — as counterparts to the full-text baselines.

Two additional lexical baselines that use **only the extracted event trigger words**, not the full text. For each summary we build one pseudo-document by concatenating its event triggers in textual order, keeping repeats (e.g. `flying. arriving. going. killing. flying.`), and vectorize it with the **same** `CountVectorizer` / `TfidfVectorizer` configuration (and `lemma_tokenizer`) used for the full-text baselines. They are stored as `baselines["bow_events"]` and `baselines["tfidf_events"]`. 


In [6]:
def compute_event_baselines(experiment_dir):
    # Load this arm's rows (full rows, embeddings included, so the in-place rewrite below keeps them).
    rows = [json.loads(l) for l in open(experiment_dir / "experiment.jsonl", encoding="utf-8") if l.strip()]

    # Since this cell needs the previous baselines cells to run, check whether tokenizer is imported already
    assert "lemma_tokenizer" in globals(), "Run the §7 full-text baseline cell first (defines lemma_tokenizer)."

    # (Again) define the experimental paths so that each cell can use a single source of truth in case the Jupyter session /terminal is disconnected throughout experiment. 
    EXPERIMENT_PATH = experiment_dir / "experiment.jsonl"
    VOCAB_EV_PATH   = experiment_dir / "baselines" / "vocab_events.json"
    VOCAB_EV_PATH.parent.mkdir(parents=True, exist_ok=True)

    # One pseudo-document per summary: trigger words in textual order, repeats kept, separated with period. Ex: "runs. climbs. knows. flies." 
    event_docs = [". ".join(e["trigger"].lower().strip() for e in r["events"]) + "." for r in rows]

    # Same vectorizer config as the full-text baselines; only the input differs (re-fit on trigger docs).
    bow_vec_ev   = CountVectorizer(tokenizer=lemma_tokenizer, token_pattern=None)
    tfidf_vec_ev = TfidfVectorizer(tokenizer=lemma_tokenizer, token_pattern=None)
    X_bow_ev   = normalize(bow_vec_ev.fit_transform(event_docs).astype(np.float64), norm="l2", axis=1, copy=False)
    X_tfidf_ev = tfidf_vec_ev.fit_transform(event_docs)   # TF-IDF L2-normalizes internally

    # Write the events to vocab_events.json
    VOCAB_EV_PATH.write_text(json.dumps({
        "bow_events":   bow_vec_ev.get_feature_names_out().tolist(),
        "tfidf_events": tfidf_vec_ev.get_feature_names_out().tolist(),
    }, ensure_ascii=False), encoding="utf-8")

    def _sparse_row(matrix, i):
        """Return {indices, values} for row i of a csr_matrix."""
        row = matrix.getrow(i)
        return {"indices": row.indices.tolist(), "values": row.data.tolist()}

    # Attach the event-trigger BoW + TF-IDF vectors to each row's baselines (additive: full-text bow/tfidf untouched).
    dim_bow_ev, dim_tfidf_ev = X_bow_ev.shape[1], X_tfidf_ev.shape[1]
    for i, r in enumerate(rows):
        b = r.setdefault("baselines", {})                 # additive: keep existing full-text bow/tfidf
        b["bow_events"]   = {"model_id": "sklearn.CountVectorizer", "task": "event_triggers",
                             "dim": dim_bow_ev,   "vector": _sparse_row(X_bow_ev, i)}
        b["tfidf_events"] = {"model_id": "sklearn.TfidfVectorizer", "task": "event_triggers",
                             "dim": dim_tfidf_ev, "vector": _sparse_row(X_tfidf_ev, i)}

    # Atomic in-place rewrite (same pattern we use as Section 7 full-text baselines).
    tmp = EXPERIMENT_PATH.with_suffix(EXPERIMENT_PATH.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as fout:
        for r in rows:
            fout.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, EXPERIMENT_PATH)

    # Sparsity stats (non-zeros per row) and count of trigger-empty summaries, for the summary printout below.
    nnz_bow   = np.diff(X_bow_ev.indptr)
    nnz_tfidf = np.diff(X_tfidf_ev.indptr)
    n_empty   = int(sum(1 for d in event_docs if not d.strip(". ")))
    print(f"Event-trigger lexical baselines -> {EXPERIMENT_PATH.name}")
    print(f"  Rows:                  {len(rows):,}   (empty trigger docs: {n_empty})")
    print(f"  BoW(events)   vocab:   {dim_bow_ev:>7,}   nnz mean={nnz_bow.mean():.1f}   max={nnz_bow.max()}")
    print(f"  TFIDF(events) vocab:   {dim_tfidf_ev:>7,}   nnz mean={nnz_tfidf.mean():.1f}   max={nnz_tfidf.max()}")
    print(f"  Vocab saved to:        {VOCAB_EV_PATH}")


for arm_dir in (NONANON_DIR, ANON_DIR):
    compute_event_baselines(arm_dir)


Event-trigger lexical baselines -> experiment.jsonl
  Rows:                  4,315   (empty trigger docs: 0)
  BoW(events)   vocab:     5,294   nnz mean=31.6   max=102
  TFIDF(events) vocab:     5,294   nnz mean=31.6   max=102
  Vocab saved to:        /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/experiments/experiment_test_9385_20260609_1013/baselines/vocab_events.json
Event-trigger lexical baselines -> experiment.jsonl
  Rows:                  4,315   (empty trigger docs: 0)
  BoW(events)   vocab:     5,182   nnz mean=30.1   max=94
  TFIDF(events) vocab:     5,182   nnz mean=30.1   max=94
  Vocab saved to:        /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/experiments/anon_experiment_test_9385_20260609_1013/baselines/vocab_events.json


**Goal:** Spot-check event-trigger baselines: for a few random summaries, show the trigger pseudo-document and the top-weighted BoW and TF-IDF terms, to confirm they were built correctly. 

In [8]:
# Choose which arm's examples to preview
ARM = NONANON_DIR        # NONANON_DIR or ANON_DIR
SAMPLE_SEED = 12        # change to see different samples
N_SAMPLES   = 3         # number of summaries to inspect
TOP_K       = 8         # number of top non-zero entries to show per vector
PREVIEW     = 400       # characters of the trigger pseudo-document to show

# Load the selected arm's rows (stream, drop embeddings so the multi-GB file does not blow up memory)
rows = [{k: v for k, v in json.loads(l).items() if k != "embeddings"}
        for l in open(ARM / "experiment.jsonl", encoding="utf-8") if l.strip()]

# Event-trigger vocabularies saved by §7.1 (index -> term, per method).
vocab_ev = json.loads((ARM / "baselines" / "vocab_events.json").read_text(encoding="utf-8"))

# "Triggerize": flying. running. climbing. ...
def _trigger_doc(r):
    return ". ".join(e["trigger"].lower().strip() for e in r["events"]) + "."

rng    = random.Random(SAMPLE_SEED)
sample = rng.sample(rows, k=min(N_SAMPLES, len(rows)))

# For better visuals, print each example with a markdown format
for r in sample:
    doc = _trigger_doc(r)
    md = [f"### {r['wikidata_id']} / {r['summary_id']} ({r['lang']})  —  {len(r['events'])} events"]
    md.append(f"**event-trigger pseudo-document** (first {PREVIEW} chars):")
    md.append("```\n" + doc[:PREVIEW] + ("..." if len(doc) > PREVIEW else "") + "\n```")
    for method, label in (("bow_events", "BoW (event triggers)"), ("tfidf_events", "TF-IDF (event triggers)")):
        b = r["baselines"][method]
        pairs = sorted(zip(b["vector"]["indices"], b["vector"]["values"]), key=lambda iv: -iv[1])[:TOP_K]
        rendered = ", ".join(f"`{vocab_ev[method][i]}`={v:.4f}" for i, v in pairs)
        md.append(f"**top-{TOP_K} {label}**: {rendered}")
    display(Markdown("\n\n".join(md)))


### 798650 / it (it)  —  54 events

**event-trigger pseudo-document** (first 400 chars):

```
crimes. decides. steal. killed. escape. runs. sent. meets. run. nicknamed. nicknamed. decide. provoking. culminates. strangle. becoming. death. rapes. hearing. rape. see. escape. escape. falls. captured. manages. meet. encounter. return. convicted. rape. sent. aware. transfer. damage. implanting. explodes. hit. sentenced. end. arranged. clashes. avoid. injures. clash. fight. death. decides. kill. ...
```

**top-8 BoW (event triggers)**: `escape`=0.3273, `rape`=0.3273, `clash`=0.2182, `death`=0.2182, `decides`=0.2182, `fight`=0.2182, `meet`=0.2182, `nicknamed`=0.2182

**top-8 TF-IDF (event triggers)**: `rape`=0.3823, `clash`=0.2779, `nicknamed`=0.2733, `implanting`=0.2129, `escape`=0.2008, `run`=0.1818, `provoking`=0.1794, `resisting`=0.1794

### 26921375 / fr (fr)  —  65 events

**event-trigger pseudo-document** (first 400 chars):

```
robbery. away. arrested. denounce. died. injuries. remains. served. sentence. allowed. works. runs. becomes. blends. seduces. released. determined. start. finds. changed. understands. suggested. move. said. uses. beats. tells. killed. lose. give. gives. died. knows. heard. meet. utter. talk. gives. grabs. kills. search. get. became. confronted. says. beating. hit. implicated. cited. kill. kills. f...
```

**top-8 BoW (event triggers)**: `kill`=0.4339, `give`=0.3254, `died`=0.2169, `allowed`=0.1085, `appear`=0.1085, `arrested`=0.1085, `arrives`=0.1085, `asks`=0.1085

**top-8 TF-IDF (event triggers)**: `kill`=0.2736, `utter`=0.2118, `implicated`=0.2118, `give`=0.2099, `blend`=0.1987, `cited`=0.1901, `died`=0.1854, `seduces`=0.1723

### 43303311 / de (de)  —  39 events

**event-trigger pseudo-document** (first 400 chars):

```
see. murder. shot. fleeing. jumps. present. focuses. earn. cutting. reports. auctioned. sale. paid. arrive at. inspect. impaled. show. murder. offer. leading to. capture. investigating. gains. access. open. employees. find. rented. break. conclude. cracked. drive. going. locks. finds. wounded. break. manage. overpower.
```

**top-8 BoW (event triggers)**: `break`=0.2981, `find`=0.2981, `murder`=0.2981, `access`=0.1491, `arrive`=0.1491, `auctioned`=0.1491, `capture`=0.1491, `conclude`=0.1491

**top-8 TF-IDF (event triggers)**: `cracked`=0.2429, `rented`=0.2429, `inspect`=0.2361, `auctioned`=0.2306, `employee`=0.2259, `impaled`=0.2048, `murder`=0.1991, `overpower`=0.1971

### 3. Cosine similarity

**Goal:** Build the pairwise similarity matrix for every encoder-condition and every lexical baseline, for both arms (non-pseudonymized and pseudonymized). These matrices are the substrate that all later per-arm retrieval metrics are computed from.

**How:** A helper function does this for one arm: stack each representation's vectors into one matrix and take `X @ X.T`. Every dense embedding and sparse baseline vector is already L2-normalized, so this dot product is the cosine similarity. The diagonal is set to negative infinity so a summary never retrieves itself. We run the function for both arms, so each arm's matrices are saved to its own `similarities.npz` (one in the non-pseudonymized folder, one in the pseudonymized folder).

In [9]:
# Compute every pairwise cosine-similarity matrix for ONE arm and save them to that arm's similarities.npz.
# Vectors are already L2-normalized, so vectors @ vectors.T is the cosine similarity; the diagonal is set
# to -inf so a summary is never retrieved as its own neighbor.
def compute_similarities(experiment_dir):
    experiment_path   = experiment_dir / "experiment.jsonl"
    similarities_path = experiment_dir / "similarities.npz"
    similarities_path.parent.mkdir(parents=True, exist_ok=True)

    # Load this arm's experiment data
    rows = [json.loads(line) for line in experiment_path.read_text(encoding="utf-8").splitlines() if line]
    n_summaries = len(rows)

    # One similarity matrix per "<encoder>__<condition>" (dense) and per "<baseline>__raw_text" (sparse)
    similarity_matrices: dict[str, np.ndarray] = {}

    # Dense encoders (e5_mistral, qwen3_emb_0p6b, story_emb): stack the vectors and take vectors @ vectors.T
    encoders = sorted({encoder for row in rows for encoder in row.get("embeddings", {})})
    for encoder in encoders:
        conditions = sorted(rows[0]["embeddings"][encoder]["vectors"].keys())
        for condition in conditions:
            vectors = np.asarray(
                [row["embeddings"][encoder]["vectors"][condition] for row in rows],
                dtype=np.float32,
            )
            similarity = vectors @ vectors.T
            np.fill_diagonal(similarity, -np.inf)
            similarity_matrices[f"{encoder}__{condition}"] = similarity

    # Sparse lexical baselines (bow, tfidf, and their event-trigger variants): rebuild the CSR matrix first
    baselines = sorted({baseline for row in rows for baseline in row.get("baselines", {})})
    for baseline in baselines:
        vocab_size = rows[0]["baselines"][baseline]["dim"]
        values, indices, index_pointer = [], [], [0]
        for row in rows:
            vector = row["baselines"][baseline]["vector"]
            values.extend(vector["values"])
            indices.extend(vector["indices"])
            index_pointer.append(len(values))
        matrix = sparse.csr_matrix((values, indices, index_pointer), shape=(n_summaries, vocab_size), dtype=np.float32)
        similarity = (matrix @ matrix.T).toarray()
        np.fill_diagonal(similarity, -np.inf)
        similarity_matrices[f"{baseline}__raw_text"] = similarity

    # Save all matrices for this arm
    np.savez_compressed(similarities_path, **similarity_matrices)

    # Quick diagnostics: off-diagonal mean per matrix (the diagonal is -inf, so mask it out)
    print(f"[{experiment_dir.name}] wrote {len(similarity_matrices)} similarity matrices to {similarities_path.name}  (N={n_summaries})")
    for name, similarity in similarity_matrices.items():
        off_diagonal = similarity[np.isfinite(similarity)]
        print(f"  {name:48s}  shape={similarity.shape}  mean_off_diag={off_diagonal.mean():.4f}  max={off_diagonal.max():.4f}")

# Run for BOTH arms so each one gets its own similarities.npz (independent inputs to the metrics step).
for arm_dir in (NONANON_DIR, ANON_DIR):
    compute_similarities(arm_dir)

[experiment_test_9385_20260609_1013] wrote 17 similarity matrices to similarities.npz  (N=4315)
  e5_mistral__causal                                shape=(4315, 4315)  mean_off_diag=0.9140  max=0.9956
  e5_mistral__events_only                           shape=(4315, 4315)  mean_off_diag=0.8890  max=0.9958
  e5_mistral__raw_text                              shape=(4315, 4315)  mean_off_diag=0.5792  max=0.9929
  e5_mistral__temporal                              shape=(4315, 4315)  mean_off_diag=0.9070  max=0.9944
  e5_mistral__temporal_causal_independent           shape=(4315, 4315)  mean_off_diag=0.9237  max=0.9943
  e5_mistral__temporal_causal_joint                 shape=(4315, 4315)  mean_off_diag=0.9169  max=0.9943
  qwen3_emb_0p6b__causal                            shape=(4315, 4315)  mean_off_diag=0.6502  max=0.9839
  qwen3_emb_0p6b__events_only                       shape=(4315, 4315)  mean_off_diag=0.6183  max=0.9812
  qwen3_emb_0p6b__raw_text                          shape=(4315,

### 4. Overall Retrieval Performance for Matched Data (non-pseudonymized vs pseudonymized)

**Goal:** Produce the thesis Table 2: retrieval performance for every baseline and structural condition, with the non-pseudonymized and pseudonymized arms side by side, on the matched set I'.

**How:** Both `experiment.jsonl` files are already restricted to I' (Step 1) and each arm's similarity matrices are already saved (Step 3). So this step only loads each arm's `similarities.npz`, builds the gold neighbors (other summaries of the same work), averages the per-query retrieval metrics (P@1, Hits@10, R-Precision, MAP, NDCG), and lays the two arms side by side. Nothing is recomputed from scratch, so it runs in seconds.

In [10]:
# Retrieval metrics for one query's ranking row. gold_neighbors = indices of other summaries of the same work.
def score_query(similarity_row, gold_neighbors):
    if not gold_neighbors: return None
    n_gold = len(gold_neighbors); gold_set = set(gold_neighbors)
    order = np.argsort(-similarity_row, kind="stable")
    relevance = np.fromiter((1 if candidate in gold_set else 0 for candidate in order), dtype=np.int64, count=len(order))
    hits = np.cumsum(relevance); precision_at_k = hits / np.arange(1, len(relevance) + 1)
    average_precision = float((precision_at_k * relevance).sum() / n_gold)
    discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))
    dcg = float((relevance * discounts).sum()); idcg = float(discounts[:n_gold].sum())
    return dict(p_at_1=float(relevance[0]), hits_at_10=float(relevance[:10].sum() > 0),
                r_precision=float(hits[n_gold-1] / n_gold), ap=average_precision, ndcg=dcg / idcg if idcg > 0 else 0.0)

# Average the five metrics over all queries, for every matrix in this arm's similarities.npz.
def evaluate_arm(experiment_dir):
    rows = [json.loads(line) for line in (experiment_dir / "experiment.jsonl").read_text(encoding="utf-8").splitlines() if line]
    work_ids = [row["wikidata_id"] for row in rows]; n_summaries = len(work_ids)
    # gold neighbors per query (same wikidata_id, excluding self); row order matches similarities.npz
    summaries_by_work = collections.defaultdict(list)
    for index, work_id in enumerate(work_ids): summaries_by_work[work_id].append(index)
    gold_neighbors = {query: [other for other in summaries_by_work[work_ids[query]] if other != query]
                      for query in range(n_summaries)}
    similarity_matrices = np.load(experiment_dir / "similarities.npz")   # diagonals are already -inf (Step 3)
    arm_metrics = {}
    for matrix_key in similarity_matrices.files:
        similarity = similarity_matrices[matrix_key]
        per_query = [metrics for query in range(n_summaries)
                     if (metrics := score_query(similarity[query], gold_neighbors[query])) is not None]
        arm_metrics[matrix_key] = {metric_name: float(np.mean([result[metric_name] for result in per_query]))
                                   for metric_name in ("p_at_1", "hits_at_10", "r_precision", "ap", "ndcg")}
    return arm_metrics

print("scoring both arms from their saved similarities.npz ...")
results_nonanon = evaluate_arm(NONANON_DIR)
results_anon    = evaluate_arm(ANON_DIR)

# Assemble Table 2 in the exact thesis row order.
table_rows = [
    ("BoW (raw text)",                "Count vectorizer",  "bow__raw_text"),
    ("TF-IDF (raw text)",             "TF-IDF vectorizer", "tfidf__raw_text"),
    ("BoW (event triggers)",          "Count vectorizer",  "bow_events__raw_text"),
    ("TF-IDF (event triggers)",       "TF-IDF vectorizer", "tfidf_events__raw_text"),
    ("Raw text",                      "Qwen3-0.6B",        "qwen3_emb_0p6b__raw_text"),
    ("Raw text",                      "E5-Mistral",        "e5_mistral__raw_text"),
    ("Raw text",                      "StoryEmbed",        "story_emb__raw_text"),
    ("Event-only",                    "Qwen3-0.6B",        "qwen3_emb_0p6b__events_only"),
    ("Event + temporal",             "Qwen3-0.6B",        "qwen3_emb_0p6b__temporal"),
    ("Event + causal",               "Qwen3-0.6B",        "qwen3_emb_0p6b__causal"),
    ("Event + temporal + causal",    "Qwen3-0.6B",        "qwen3_emb_0p6b__temporal_causal_independent"),
    ("Event + joint temporo-causal", "Qwen3-0.6B",        "qwen3_emb_0p6b__temporal_causal_joint"),
    ("Event-only",                    "E5-Mistral",        "e5_mistral__events_only"),
    ("Event + temporal",             "E5-Mistral",        "e5_mistral__temporal"),
    ("Event + causal",               "E5-Mistral",        "e5_mistral__causal"),
    ("Event + temporal + causal",    "E5-Mistral",        "e5_mistral__temporal_causal_independent"),
    ("Event + joint temporo-causal", "E5-Mistral",        "e5_mistral__temporal_causal_joint"),
]
metric_columns = [("p_at_1", "P@1"), ("hits_at_10", "Hits@10"), ("r_precision", "R-Prec."), ("ap", "MAP"), ("ndcg", "NDCG")]
table_data = []
for representation, encoder, matrix_key in table_rows:
    row = [representation, encoder]
    for arm_metrics in (results_nonanon, results_anon):
        scores = arm_metrics.get(matrix_key, {})
        row += [scores.get(metric_key, float("nan")) for metric_key, _ in metric_columns]
    table_data.append(row)
columns = pd.MultiIndex.from_tuples(
    [("", "Representation"), ("", "Vectorizer/Encoder")] +
    [("Non-Pseudonymized", metric_label) for _, metric_label in metric_columns] +
    [("Pseudonymized", metric_label) for _, metric_label in metric_columns])
matched_df = pd.DataFrame(table_data, columns=columns)

n_queries = sum(1 for line in open(NONANON_DIR / "experiment.jsonl", encoding="utf-8") if line.strip())
print(f"\nTable 2: matched retrieval over I' = {n_queries} queries (both arms), each with >=1 gold neighbor\n")
with pd.option_context("display.max_columns", None, "display.width", 240):
    print(matched_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
# matched_df holds all five metrics for both arms -> format / bold-underline for the thesis as needed.

scoring both arms from their saved similarities.npz ...

Table 2: matched retrieval over I' = 4315 queries (both arms), each with >=1 gold neighbor

                                                Non-Pseudonymized                               Pseudonymized                              
              Representation Vectorizer/Encoder               P@1 Hits@10 R-Prec.    MAP   NDCG           P@1 Hits@10 R-Prec.    MAP   NDCG
              BoW (raw text)   Count vectorizer            0.9154  0.9740  0.8637 0.8990 0.9312        0.0732  0.1638  0.0579 0.0801 0.2273
           TF-IDF (raw text)  TF-IDF vectorizer            0.9374  0.9859  0.8958 0.9265 0.9512        0.0811  0.1900  0.0668 0.0996 0.2595
        BoW (event triggers)   Count vectorizer            0.2058  0.3235  0.1503 0.1748 0.3138        0.1919  0.3015  0.1390 0.1619 0.2997
     TF-IDF (event triggers)  TF-IDF vectorizer            0.2440  0.4097  0.1844 0.2147 0.3568        0.2250  0.3815  0.1674 0.1973 0.3387
           

### 5. Incremental contribution of Temporal-Causal Structural Enrichments
**Goal:** For each encoder, show how each retrieval metric changes when relation structure is added on top of the event-only representation, with the non-pseudonymized and pseudonymized arms side by side.

In [ ]:

# Check if the previous prerequisite cell is ran first
assert "results_nonanon" in globals() and "results_anon" in globals(), "Run Step 4 (Table 2) first; it computes results_nonanon / results_anon."

# The two encoders, the four added-structure conditions, and the five metrics (with their delta labels).
encoders = [("Qwen3-0.6B", "qwen3_emb_0p6b"), ("E5-Mistral", "e5_mistral")]
structures = [("Temporal",             "temporal"),
              ("Causal",               "causal"),
              ("Temporal + causal",    "temporal_causal_independent"),
              ("Joint temporo-causal", "temporal_causal_joint")]
metric_columns = [("p_at_1", "ΔP@1"), ("hits_at_10", "ΔHits@10"), ("r_precision", "ΔR-Prec."),
                  ("ap", "ΔMAP"), ("ndcg", "ΔNDCG")]

# Change in a metric from adding structure, vs the same encoder's events_only, within one arm
def structure_delta(arm_metrics, encoder, condition, metric_key):
    return (arm_metrics.get(f"{encoder}__{condition}", {}).get(metric_key, float("nan"))
            - arm_metrics.get(f"{encoder}__events_only", {}).get(metric_key, float("nan")))

# One row per (encoder, added structure): the deltas for the non-pseudonymized arm, then the pseudonymized arm
table_data = []
for encoder_label, encoder in encoders:
    for structure_label, condition in structures:
        row = [encoder_label, structure_label]
        for arm_metrics in (results_nonanon, results_anon):
            row += [structure_delta(arm_metrics, encoder, condition, metric_key) for metric_key, _ in metric_columns]
        table_data.append(row)

# Two-level header: a Non-Pseudonymized block and a Pseudonymized block, each with the five delta metrics
columns = pd.MultiIndex.from_tuples(
    [("", "Encoder"), ("", "Added structure")] +
    [("Non-Pseudonymized", metric_label) for _, metric_label in metric_columns] +
    [("Pseudonymized", metric_label) for _, metric_label in metric_columns])
ablation_df = pd.DataFrame(table_data, columns=columns)

# Print the table (positive = structure improves retrieval; negative = it degrades)
print("Incremental retrieval change from adding relation structure to event-only (matched I')")
print("(delta = condition - events_only, same encoder and arm; + = improvement)\n")
with pd.option_context("display.max_columns", None, "display.width", 320):
    print(ablation_df.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

### 6. Event-set retrieval: the Tversky diagnostic

**Goal:** Ask whether event overlap *alone* can retrieve other summaries of the same work, with no relations and no embeddings. Each summary is represented purely as a set of its extracted events.

**Two set representations:**
- **Event types:** the set of distinct MAVEN categories (abstract; e.g. {Attack, Statement, Motion}).
- **Event triggers:** the set of distinct trigger words (lexical; e.g. {attacked, killed, fled}).

**Similarity:** the symmetric Tversky index `T(A, B) = |A ∩ B| / (|A ∩ B| + α|A \ B| + β|B \ A|)` with `α = β = 0.5`, which equals the Dice coefficient.

**How:** Reuse the matched set I' from Step 1 (no re-intersection). For each arm, read the extracted events from `tma_subset_events_processed.test.jsonl`, build the two set representations, score them with the same per-query retrieval protocol used for Table 2, and place the non-pseudonymized and pseudonymized blocks side by side.



In [ ]:

# Reuse the matched set built in Step 1 (no re-intersection); sort for a deterministic row order.
assert "Iprime" in globals(), "Run Step 1 first; it builds the matched set Iprime."
matched_keys = sorted(Iprime)
ALPHA = BETA = 0.5                       # symmetric Tversky == Dice coefficient
id_pattern = re.compile(r'"wikidata_id":\s*"([^"]*)",\s*"summary_id":\s*"([^"]*)"')

# Per-query retrieval metrics for one ranking row (gold = other summaries of the same work).
def query_metrics(similarity_row, gold_neighbors):
    if not gold_neighbors: return None
    n_gold = len(gold_neighbors); gold_set = set(gold_neighbors)
    order = np.argsort(-similarity_row, kind="stable")
    relevance = np.fromiter((1 if candidate in gold_set else 0 for candidate in order), dtype=np.int64, count=len(order))
    hits = np.cumsum(relevance); precision_at_k = hits / np.arange(1, len(relevance) + 1)
    average_precision = float((precision_at_k * relevance).sum() / n_gold)
    discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))
    dcg = float((relevance * discounts).sum()); idcg = float(discounts[:n_gold].sum())
    return {"P@1": float(relevance[0]), "Hits@10": float(relevance[:10].sum() > 0), "R-Prec.": float(hits[n_gold-1] / n_gold),
            "MAP": average_precision, "NDCG": dcg / idcg if idcg > 0 else 0.0}

# Score one arm: read its extracted events for the matched keys, build two set representations, rank by Tversky.
def tversky_stats(events_path, keys):
    # Events files are far smaller than experiment.jsonl, so read them directly and keep only matched keys.
    key_to_index = {key: index for index, key in enumerate(keys)}; key_set = set(keys); n_summaries = len(keys)
    work_ids = [None] * n_summaries; events_per_summary = [None] * n_summaries
    with open(events_path, encoding="utf-8") as events_file:
        for line in events_file:
            match = id_pattern.search(line[:160])
            if not match: continue
            key = (match.group(1), match.group(2))
            if key not in key_set: continue
            record = json.loads(line); index = key_to_index[key]
            work_ids[index] = key[0]; events_per_summary[index] = record.get("events", [])

    # Two set representations per summary: abstract MAVEN categories, and lexical trigger words.
    representations = {
        "Event types (MAVEN categories)": [{event["event_type"] for event in events} for events in events_per_summary],
        "Events (trigger words)":         [{event["trigger"].lower().strip() for event in events} for events in events_per_summary],
    }

    # Gold neighbors per query (same wikidata_id, excluding self).
    summaries_by_work = collections.defaultdict(list)
    for index, work_id in enumerate(work_ids): summaries_by_work[work_id].append(index)
    gold_neighbors = {query: [other for other in summaries_by_work[work_ids[query]] if other != query]
                      for query in range(n_summaries)}

    metric_names = ["P@1", "Hits@10", "R-Prec.", "MAP", "NDCG"]
    representation_stats = {}
    for representation_name, summary_sets in representations.items():
        # One-hot the set membership, then compute the symmetric Tversky similarity over all pairs (formula in the markdown above).
        vocabulary = sorted({item for summary_set in summary_sets for item in summary_set})
        vocab_index = {item: position for position, item in enumerate(vocabulary)}
        membership = np.zeros((n_summaries, len(vocabulary)), dtype=np.float32)
        for index, summary_set in enumerate(summary_sets):
            for item in summary_set: membership[index, vocab_index[item]] = 1.0
        intersection = membership @ membership.T
        set_sizes = membership.sum(1); size_a, size_b = set_sizes[:, None], set_sizes[None, :]
        similarity = (intersection / (intersection + ALPHA * (size_a - intersection) + BETA * (size_b - intersection))).astype(np.float32)
        np.fill_diagonal(similarity, -np.inf)
        # Average the metrics over queries, plus descriptive set-size and similarity stats.
        per_query = [metrics for query in range(n_summaries)
                     if (metrics := query_metrics(similarity[query], gold_neighbors[query])) is not None]
        item_counts = np.array([len(summary_set) for summary_set in summary_sets])
        off_diagonal = similarity[np.isfinite(similarity)]
        representation_stats[representation_name] = {
            "Vocab size": len(vocabulary), "Mean items": float(item_counts.mean()), "SD items": float(item_counts.std()),
            "Min": int(item_counts.min()), "Max": int(item_counts.max()),
            "Mean Sim.": float(off_diagonal.mean()), "SD Sim.": float(off_diagonal.std()),
            **{metric_name: float(np.mean([result[metric_name] for result in per_query])) for metric_name in metric_names}}
    return representation_stats

# Score both arms on the SAME matched keys.
print(f"computing the Tversky diagnostic on the matched set I' = {len(matched_keys)} summaries ...")
stats_nonanon = tversky_stats(NONANON_DIR / "tma_subset_events_processed.test.jsonl", matched_keys)
stats_anon    = tversky_stats(ANON_DIR    / "tma_subset_events_processed.test.jsonl", matched_keys)

# Assemble the table: a Non-Pseudonymized block and a Pseudonymized block, identical columns.
descriptive_columns = ["Vocab size", "Mean items", "SD items", "Min", "Max", "Mean Sim.", "SD Sim."]
metric_columns = ["P@1", "Hits@10", "R-Prec.", "MAP", "NDCG"]
formats = {"Vocab size": "{:.0f}", "Mean items": "{:.1f}", "SD items": "{:.1f}", "Min": "{:.0f}", "Max": "{:.0f}",
           "Mean Sim.": "{:.3f}", "SD Sim.": "{:.3f}", **{column: "{:.4f}" for column in metric_columns}}

# One block (two rows) per arm, formatted for display.
def build_block(condition, representation_stats):
    rows = []
    for representation_name in ["Event types (MAVEN categories)", "Events (trigger words)"]:
        values = representation_stats[representation_name]
        row = {"Condition": condition, "Story representation": representation_name}
        row.update({column: formats[column].format(values[column]) for column in descriptive_columns + metric_columns})
        rows.append(row)
    return rows

table_rows = build_block("Non-Pseudonymized", stats_nonanon) + build_block("Pseudonymized", stats_anon)
tversky_df = pd.DataFrame(table_rows)[["Condition", "Story representation"] + descriptive_columns + metric_columns]
print(f"\nSet-based Tversky index (a=b={ALPHA}, Dice) | matched I' = {len(matched_keys)} summaries / {len({work_id for work_id, _ in matched_keys})} stories\n")
with pd.option_context("display.max_columns", None, "display.width", 240):
    print(tversky_df.to_string(index=False))